In [1]:
# This cell runs first and fixes a JupyterLab display issue.
# Without it, each cell's output gets trapped inside a small scrollable box,
# which makes the tool look broken. This removes those boxes so the UI
# spreads across the full page like a normal website.

from IPython.display import display, HTML
display(HTML("""
<style>
  /* Force all output areas to expand fully — no scroll boxing */
  .jp-OutputArea-output {
    max-height: none !important;
    overflow: visible !important;
  }
  .jp-Cell-outputWrapper {
    max-height: none !important;
    overflow: visible !important;
  }
  .jp-OutputArea {
    max-height: none !important;
    overflow: visible !important;
  }
  /* Remove the scroll indicator arrows */
  .jp-OutputArea-prompt {
    display: none !important;
  }
  /* Remove cell borders and padding that create the box look */
  .jp-Cell {
    padding: 0 !important;
    border: none !important;
  }
  .jp-Cell-outputArea {
    padding: 0 !important;
  }
</style>
"""))

In [2]:
# This cell installs the three external packages the tool needs.
# - anthropic: lets us talk to the Claude AI API to find matching quotes
# - python-dotenv: reads the API key from the .env file so it stays private
# - pypdf: reads and extracts text from uploaded PDF transcripts
# The -q flag keeps the install output quiet so it doesn't clutter the screen.
# Run this once — you don't need to run it again unless you reset the environment.

!pip install anthropic python-dotenv pypdf -q

In [3]:
# This cell loads all the tools and libraries the notebook will use,
# and reads the Anthropic API key from the .env file.
# Every other cell depends on these being loaded first, so this must run
# before anything else.

import anthropic          # Talks to the Claude AI API
import re                 # Helps search for patterns in text (like speaker labels)
import json               # Converts data between Python and the format the API uses
import ipywidgets as widgets  # Builds the interactive UI elements (buttons, sliders, etc.)
from IPython.display import display, HTML, clear_output  # Controls what gets shown on screen
from dotenv import load_dotenv  # Reads the .env file to get the API key
import os                 # Lets Python read environment variables like the API key
import ipywidgets         # Imported again to ensure version compatibility

# Load the API key from the .env file into the environment.
# The __ = trick suppresses the "True" return value that would otherwise print.
__ = load_dotenv()

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# SECURITY NOTE
# Your transcript is sent over HTTPS (encrypted) to Anthropic's API,
# processed to find matching quotes, and the results are returned.
# Anthropic does not use API data for training by default.
# The transcript is never saved to disk — it only lives in memory while
# the notebook is running, and disappears when the kernel is restarted.
# Full privacy policy: https://www.anthropic.com/privacy
# ─────────────────────────────────────────────────────────────────────────────

import re

# ── SPEAKER LABEL SCANNER ─────────────────────────────────────────────────────
# Looks through the transcript for lines that look like speaker labels
# (e.g. "Interviewer:", "Speaker 1:", "[P2]:") and returns a list of names.
# Used by the speaker filter checkboxes in the UI.

def detect_speakers(transcript: str) -> list[str]:
    speakers = set()
    pattern = re.compile(
        r'^\s*(\[?[A-Za-z][A-Za-z0-9 _\-]{0,40}\]?)\s*:',
        re.MULTILINE
    )
    # Words that look like speaker labels but aren't — skip these
    false_positives = {
        'http', 'https', 'note', 'edit', 'update', 'timestamp',
        'date', 'time', 'location', 'subject', 'from', 'to', 'cc'
    }
    for match in pattern.finditer(transcript):
        label = match.group(1).strip().strip('[]')
        if label.lower() not in false_positives and len(label) >= 2:
            speakers.add(label)
    return sorted(speakers)


# ── SPEAKER FILTER ────────────────────────────────────────────────────────────
# When the researcher checks specific speakers, this trims the transcript
# down to only those speakers' lines before sending it to the AI.
# This means the AI never even sees lines from speakers you didn't select,
# which keeps results focused and avoids the interviewer's questions
# showing up as matches.

def filter_transcript_by_speakers(transcript: str, selected_speakers: list[str]) -> str:
    # If no speakers are selected, return the full transcript unchanged
    if not selected_speakers:
        return transcript

    lines      = transcript.split('\n')
    result     = []
    capturing  = False
    # Pattern that matches lines starting with one of the selected speaker names
    speaker_re = re.compile(
        r'^\s*(?:\[)?(' + '|'.join(re.escape(s) for s in selected_speakers) + r')(?:\])?\s*:?\s',
        re.IGNORECASE
    )
    # Pattern that matches any speaker label (used to detect when a new speaker starts)
    any_speaker_re = re.compile(r'^\s*([A-Za-z][A-Za-z0-9 _\-]{0,30}):\s')

    for line in lines:
        if speaker_re.match(line):
            # This line belongs to a selected speaker — start capturing
            capturing = True
            result.append(line)
        elif any_speaker_re.match(line):
            # This line belongs to a different speaker — stop capturing
            capturing = False
        elif capturing:
            # This is a continuation line from the selected speaker — keep it
            result.append(line)

    return '\n'.join(result)


# ── MAIN QUOTE SEARCH FUNCTION ────────────────────────────────────────────────
# This is the heart of the tool. It takes the researcher's description
# (which can be vague), filters the transcript to the right speakers,
# sends everything to Claude AI, and gets back a ranked list of verbatim
# quotes that best match what the researcher described.
# Results are then filtered by the minimum score the researcher set,
# and sorted from highest to lowest match score.

def extract_quotes_from_transcript(
    transcript: str,
    researcher_description: str,
    min_score: int = 0,
    selected_speakers: list[str] = None
) -> list[dict]:
    import anthropic, json
    from dotenv import load_dotenv
    import os
    load_dotenv()

    # Trim the transcript to only the selected speakers' lines
    filtered = filter_transcript_by_speakers(transcript, selected_speakers or [])
    if not filtered.strip():
        return []

    # Connect to the Claude API using the key from the .env file
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    # Send the transcript and description to Claude with strict instructions:
    # - Only return exact verbatim text, never paraphrase
    # - Score each quote 1-100 for how well it matches the description
    # - Include timestamps, speaker labels, and key terms if present
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2000,
        system="""You are a UX research assistant helping researchers find verbatim quotes from interview transcripts.

Rules:
1. Understand the INTENT and SENTIMENT behind the researcher's description — it may be vague or paraphrased.
2. Return ONLY verbatim text copied exactly from the transcript — never paraphrase, never alter a single word.
3. If a timestamp (e.g. [00:12], 0:45, 1:23:45) appears near a quote in the transcript, capture it in the timestamp field.
4. Assign a match_score from 1-100 reflecting how closely the quote matches the described sentiment (100 = perfect match).
5. List key_terms: the specific words or short phrases in the quote that most strongly signal the match.
6. If a speaker label is present before the quote (e.g. "Speaker 1:", "Interviewer:"), capture it in the speaker field.
7. Rank results by match_score descending.
8. Return as many strong matches as you can find — the caller will filter by minimum score.

Respond ONLY in this exact JSON format, no extra text, no markdown fences:
{
  "results": [
    {
      "quote": "<exact verbatim text>",
      "timestamp": "<timestamp or null>",
      "reasoning": "<one sentence>",
      "match_score": <integer 1-100>,
      "key_terms": ["<term1>", "<term2>"],
      "speaker": "<speaker label or null>"
    }
  ]
}""",
        messages=[
            {
                "role": "user",
                "content": f"""Researcher's description:
\"\"\"{researcher_description}\"\"\"

Transcript (pre-filtered to relevant speakers):
\"\"\"{filtered}\"\"\"

Return all relevant verbatim quotes ranked by match_score. JSON only, no markdown."""
            }
        ]
    )

    # Parse the AI's JSON response, stripping any accidental code fences
    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    parsed = json.loads(raw)
    results = parsed.get("results", [])

    # Keep only results that meet the minimum score, then sort highest first
    results = [r for r in results if int(r.get("match_score", 0)) >= min_score]
    results = sorted(results, key=lambda r: int(r.get("match_score", 0)), reverse=True)
    return results

In [5]:
# This cell defines four small helper functions used by the results display.
# None of these talk to the AI — they're purely about how text is processed
# and displayed on screen.


# ── KEY TERM HIGHLIGHTER ──────────────────────────────────────────────────────
# Takes a quote and a list of key terms, and wraps each term in a yellow
# highlight mark so they stand out visually in the result cards.

def highlight_terms(text: str, terms: list[str]) -> str:
    if not terms:
        return text
    # Sort longest terms first to avoid accidentally highlighting part of a longer term
    for term in sorted(terms, key=len, reverse=True):
        pattern = re.compile(re.escape(term), re.IGNORECASE)
        text = pattern.sub(
            lambda m: f'<mark style="background:#fff176;border-radius:3px;padding:0 2px;">{m.group()}</mark>',
            text
        )
    return text


# ── QUOTE LOCATOR ─────────────────────────────────────────────────────────────
# Finds where a quote appears in the transcript by its character position.
# Tries several increasingly fuzzy methods in case the AI returned the quote
# with slightly different spacing or punctuation than the original text.
# Returns the character position, or -1 if the quote can't be found.

def find_quote_char_offset(transcript: str, quote: str) -> int:
    if not quote:
        return -1

    # Try 1: exact match — works most of the time
    idx = transcript.find(quote)
    if idx != -1:
        return idx

    # Try 2: ignore extra whitespace differences
    normalized_quote      = " ".join(quote.split())
    normalized_transcript = " ".join(transcript.split())
    idx = normalized_transcript.find(normalized_quote)
    if idx != -1:
        return transcript.find(normalized_quote[:40])

    # Try 3: just the first 60 characters of the quote
    snippet = quote[:60].strip()
    idx = transcript.find(snippet)
    if idx != -1:
        return idx

    # Try 4: just the first 40 characters
    snippet = quote[:40].strip()
    idx = transcript.find(snippet)
    if idx != -1:
        return idx

    # Try 5: first 8 words — useful for longer quotes
    words = quote.split()
    if len(words) >= 8:
        snippet = " ".join(words[:8])
        idx = transcript.find(snippet)
        if idx != -1:
            return idx

    # Try 6: first 4 words — last resort for very short or altered quotes
    if len(words) >= 4:
        snippet = " ".join(words[:4])
        idx = transcript.find(snippet)
        if idx != -1:
            return idx

    return -1


# ── TRANSCRIPT HTML BUILDER ───────────────────────────────────────────────────
# Converts the plain transcript text into HTML where each line gets its own
# ID tag (like "tline-42"). This makes it possible to scroll and highlight
# a specific line when the researcher clicks a result card.

def build_transcript_html(transcript: str) -> str:
    lines = transcript.split("\n")
    parts = []
    char_pos = 0
    for i, line in enumerate(lines):
        # Escape HTML special characters so they display correctly
        escaped = line.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        parts.append(
            f'<span id="tline-{i}" data-charpos="{char_pos}" '
            f'style="display:block;padding:1px 0;">{escaped}</span>'
        )
        char_pos += len(line) + 1
    return "\n".join(parts)


# ── LINE ID CONVERTER ─────────────────────────────────────────────────────────
# Takes a character position (from find_quote_char_offset above) and works
# out which line number in the transcript that corresponds to.
# Returns the HTML ID of that line (e.g. "tline-42") so the page can scroll to it.

def char_offset_to_line_id(transcript: str, offset: int) -> str | None:
    if offset < 0:
        return None
    lines = transcript.split("\n")
    pos = 0
    for i, line in enumerate(lines):
        if pos + len(line) >= offset:
            return f"tline-{i}"
        pos += len(line) + 1
    return None

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# This is the main cell that builds and displays the entire tool interface.
# It's the longest cell because it combines:
#   1. Page styling (fonts, colors, layout — the visual design)
#   2. The transcript input area (paste or upload)
#   3. The speaker filter
#   4. The search description box and score slider
#   5. The Find Quotes button and results display
#   6. The persistent quote library
# ─────────────────────────────────────────────────────────────────────────────


# ── VISUAL DESIGN + PAGE HEADER ───────────────────────────────────────────────
# Injects the page's fonts, color palette, and visual styling.
# Also displays the "Transcript Quote Finder" title at the top of the page.
# Doing this here (rather than a separate cell) ensures the title always
# appears above the UI rather than below it.

display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,300;0,400;0,500;1,300;1,400&family=Jost:wght@300;400;500&family=DM+Mono:wght@300;400&display=swap');

  :root {
    --cream:       #f5f0e8;
    --cream-2:     #ede7d9;
    --cream-3:     #e4ddd0;
    --surface:     #faf7f2;
    --surface-2:   #f0ebe0;
    --ink:         #2c2820;
    --ink-2:       #6b6358;
    --ink-3:       #a39b8e;
    --ink-4:       #c4bdb4;
    --accent:      #8b6f47;
    --accent-soft: #f0e8d8;
    --green:       #4a6741;
    --amber:       #8b6f47;
    --red:         #8b4a4a;
    --radius:      8px;
    --radius-lg:   14px;
    --shadow-sm:   0 1px 4px rgba(44,40,32,0.06);
    --shadow-md:   0 4px 20px rgba(44,40,32,0.09);
    --serif:       'Cormorant Garamond', Georgia, serif;
    --sans:        'Jost', sans-serif;
    --mono:        'DM Mono', monospace;
    --transition:  0.2s cubic-bezier(0.4, 0, 0.2, 1);
  }

  body, .jp-Notebook, .jp-Cell, .jp-OutputArea,
  .jp-OutputArea-output, .lm-Widget {
    background: var(--cream) !important;
  }

  .widget-vbox, .widget-hbox {
    border: none !important;
    box-shadow: none !important;
    overflow: visible !important;
  }
  .jp-OutputArea-output, .jp-Cell-outputWrapper {
    overflow: visible !important;
  }

  .jp-CodeMirrorEditor, .jp-InputArea-editor,
  .jp-Cell-inputWrapper, .jp-InputPrompt, .jp-OutputPrompt {
    display: none !important;
  }

  .jp-Notebook { scroll-behavior: auto !important; }

  .widget-vbox, .widget-hbox, .widget-html,
  .widget-label, .widget-button, .widget-checkbox label {
    font-family: var(--sans) !important;
    color: var(--ink) !important;
  }

  div[style*="background:#fff8e1"],
  div[style*="background: #fff8e1"] {
    background: var(--surface-2) !important;
    border-left: 2px solid var(--accent) !important;
    border-radius: var(--radius) !important;
    color: var(--ink-2) !important;
    font-size: 11.5px !important;
    font-family: var(--sans) !important;
  }

  .widget-button {
    font-family: var(--sans) !important;
    font-size: 11.5px !important;
    font-weight: 500 !important;
    letter-spacing: 0.04em !important;
    border-radius: var(--radius) !important;
    border: 1px solid var(--cream-3) !important;
    background: var(--surface) !important;
    color: var(--ink) !important;
    box-shadow: var(--shadow-sm) !important;
    transition: all var(--transition) !important;
  }
  .widget-button:hover {
    background: var(--cream-2) !important;
    box-shadow: var(--shadow-md) !important;
    transform: translateY(-1px) !important;
  }
  .widget-button.mod-primary {
    background: var(--ink) !important;
    color: var(--cream) !important;
    border-color: var(--ink) !important;
  }
  .widget-button.mod-primary:hover { background: #3d3830 !important; }
  .widget-button.mod-success {
    background: var(--green) !important;
    color: var(--cream) !important;
    border-color: var(--green) !important;
  }
  .widget-button.mod-info {
    background: var(--surface-2) !important;
    color: var(--ink) !important;
    border-color: var(--cream-3) !important;
  }

  .widget-textarea textarea {
    font-family: var(--mono) !important;
    font-size: 12px !important;
    font-weight: 300 !important;
    border: 1px solid var(--cream-3) !important;
    border-radius: var(--radius-lg) !important;
    background: var(--surface) !important;
    color: var(--ink) !important;
    padding: 14px !important;
    line-height: 1.8 !important;
    box-shadow: var(--shadow-sm) !important;
    resize: vertical !important;
  }
  .widget-textarea textarea:focus {
    border-color: var(--accent) !important;
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(139,111,71,0.1) !important;
  }
  .widget-textarea textarea::placeholder {
    color: var(--ink-4) !important;
    font-style: italic !important;
  }

  .widget-checkbox input[type="checkbox"] { accent-color: var(--accent) !important; }
  .widget-checkbox label {
    font-size: 12.5px !important;
    color: var(--ink-2) !important;
  }

  .widget-html b {
    font-family: var(--serif) !important;
    font-weight: 500 !important;
    font-size: 17px !important;
    color: var(--ink) !important;
  }
  .widget-html span { font-family: var(--sans) !important; }

  .qcard {
    background: var(--surface) !important;
    border: 1px solid var(--cream-3) !important;
    border-radius: var(--radius-lg) !important;
    box-shadow: var(--shadow-sm) !important;
    transition: box-shadow var(--transition), transform var(--transition) !important;
    font-family: var(--sans) !important;
    overflow: hidden !important;
  }
  .qcard:hover {
    box-shadow: var(--shadow-md) !important;
    transform: translateY(-2px) !important;
  }
  .qcard > div:first-child {
    background: var(--surface-2) !important;
    border-bottom: 1px solid var(--cream-3) !important;
  }

  .preview-text, .full-text {
    font-family: var(--serif) !important;
    font-size: 15.5px !important;
    color: var(--ink) !important;
    line-height: 1.85 !important;
    font-style: italic !important;
  }

  mark {
    background: var(--accent-soft) !important;
    color: var(--accent) !important;
    border-radius: 3px !important;
    padding: 1px 3px !important;
    font-style: normal !important;
  }

  #transcript-pane {
    background: #1e1b16 !important;
    border: 1px solid #2e2a22 !important;
    border-radius: var(--radius-lg) !important;
    font-family: var(--mono) !important;
    font-size: 11.5px !important;
    color: #c8c0b0 !important;
    line-height: 1.9 !important;
    box-shadow: var(--shadow-md) !important;
  }
  #transcript-pane span.tline-highlight {
    background: #2e2a22 !important;
    outline: 1.5px solid var(--accent) !important;
    border-radius: 2px !important;
  }

  div[style*="letter-spacing:.04em"],
  div[style*="letter-spacing: .04em"] {
    font-family: var(--mono) !important;
    font-size: 9.5px !important;
    letter-spacing: 0.14em !important;
    text-transform: uppercase !important;
    color: var(--ink-3) !important;
  }

  span[style*="background:#e8f5e9"] {
    background: var(--accent-soft) !important;
    color: var(--accent) !important;
    border-radius: 20px !important;
  }
  span[style*="background:#e3f2fd"] {
    background: var(--surface-2) !important;
    color: var(--ink-2) !important;
    border-radius: 20px !important;
  }
  span[style*="background:#f3e5f5"] {
    background: var(--cream-2) !important;
    color: var(--ink-2) !important;
    border-radius: 20px !important;
  }

  .toggle-btn {
    font-family: var(--sans) !important;
    font-size: 11px !important;
    border: 1px solid var(--cream-3) !important;
    border-radius: var(--radius) !important;
    color: var(--ink-2) !important;
    background: var(--surface) !important;
    cursor: pointer !important;
  }
  .toggle-btn:hover { background: var(--cream-2) !important; }

  #quote-library-root {
    background: var(--surface) !important;
    border: 1px solid var(--cream-3) !important;
    border-radius: var(--radius-lg) !important;
    box-shadow: var(--shadow-md) !important;
    font-family: var(--sans) !important;
  }
  #quote-library-root > div:first-child {
    background: var(--ink) !important;
    color: var(--cream) !important;
    border-radius: var(--radius-lg) var(--radius-lg) 0 0 !important;
  }

  .folder-drop-zone {
    border: 1.5px dashed var(--cream-3) !important;
    border-radius: var(--radius-lg) !important;
    transition: all var(--transition) !important;
  }
  .folder-drop-zone.drag-over {
    border-color: var(--accent) !important;
    background: var(--accent-soft) !important;
  }

  .saved-quote-item {
    background: var(--surface-2) !important;
    border: 1px solid var(--cream-3) !important;
    border-radius: var(--radius) !important;
    font-family: var(--serif) !important;
  }

  .export-btn {
    border: 1px solid var(--cream-3) !important;
    color: var(--ink-2) !important;
    border-radius: var(--radius) !important;
    font-size: 10.5px !important;
    background: transparent !important;
    transition: all var(--transition) !important;
  }
  .export-btn:hover {
    background: var(--ink) !important;
    color: var(--cream) !important;
  }

  #sidebar-toggle {
    font-family: var(--sans) !important;
    font-size: 12px !important;
    background: var(--ink) !important;
    color: var(--cream) !important;
    border: none !important;
    border-radius: var(--radius) !important;
    padding: 7px 16px !important;
    box-shadow: var(--shadow-sm) !important;
    transition: all var(--transition) !important;
  }
  #sidebar-toggle:hover {
    background: #3d3830 !important;
    box-shadow: var(--shadow-md) !important;
  }

  /* Page header — the big title at the top */
  #uxr-page-header {
    padding: 48px 10px 32px 10px;
    border-bottom: 1px solid var(--cream-3);
    margin-bottom: 36px;
    max-width: 1400px;
  }
  #uxr-page-header h1 {
    font-family: var(--serif);
    font-size: 48px;
    font-weight: 300;
    color: var(--ink);
    margin: 0 0 8px 0;
    letter-spacing: -0.01em;
    line-height: 1.1;
  }
  #uxr-page-header p {
    font-family: var(--sans);
    font-size: 13px;
    font-weight: 300;
    color: var(--ink-3);
    margin: 0;
    letter-spacing: 0.02em;
  }
</style>

<script>
  (function() {
    /* Stops JupyterLab from automatically scrolling down to the last cell
       every time a cell runs — which would prevent users from scrolling up */
    const disableAutoScroll = () => {
      document.querySelectorAll('.jp-Cell').forEach(cell => {
        cell.scrollIntoView = function() {};
      });
    };
    disableAutoScroll();
    const observer = new MutationObserver(disableAutoScroll);
    observer.observe(document.body, { childList: true, subtree: true });
  })();
</script>

<!-- The page title and subtitle shown at the very top of the tool -->
<div id="uxr-page-header">
  <h1>Transcript Quote Finder</h1>
  <p>Upload an interview transcript — find verbatim quotes by describing what you're looking for</p>
</div>
"""))

# ── TRANSCRIPT SECTION HEADER + PRIVACY WARNING ───────────────────────────────
# The "Interview Transcript" label and the yellow privacy reminder banner
# that appears above the paste/upload area.

# Note: transcript_label is defined twice below — the first definition
# is a leftover placeholder from a previous edit and can be ignored.
# The second definition (immediately after) is the real one.
transcript_label = widgets.HTML("""
...rest of Cell 5 continues unchanged from here...
""")

transcript_label = widgets.HTML("""
<div style="font-family:'Segoe UI',sans-serif;">
  <b style="font-size:14px;">📄 Interview Transcript</b>
  <div style="margin-top:8px;padding:10px 14px;background:#fff8e1;border-left:4px solid #f9a825;
              border-radius:4px;font-size:12px;color:#555;line-height:1.6;">
    ⚠️ <strong>Privacy reminder:</strong> Please don't upload transcripts with personally
    identifying information for your participants. As an extra layer of security, please
    redact names, ages, and other personally identifying information on your own discretion.
    Additionally, ensure that you have disclosed to participants that AI may review
    transcript data.
  </div>
</div>
""")

# ── PASTE / UPLOAD TOGGLE BUTTONS ─────────────────────────────────────────────
# Two buttons at the top of the transcript area that switch between
# pasting text directly or uploading a file.

btn_paste  = widgets.Button(description="✏️ Paste Text",  layout=widgets.Layout(width="140px", height="32px"))
btn_upload = widgets.Button(description="📁 Upload File", layout=widgets.Layout(width="140px", height="32px"))
toggle_row = widgets.HBox([btn_paste, btn_upload], layout=widgets.Layout(margin="8px 0 6px 0"))

# The large text area where users paste their transcript directly
transcript_box = widgets.Textarea(
    placeholder="Paste your transcript here. Timestamps like [00:45] or 1:23 are supported.",
    layout=widgets.Layout(width="100%", height="220px")
)
# Wraps the textarea in a container so it can be shown/hidden as a panel
paste_panel = widgets.VBox([transcript_box])

# ── FILE UPLOAD PANEL ─────────────────────────────────────────────────────────
# The file picker (Choose file) and Load File button.
# Hidden by default — only appears when the user clicks "Upload File".

upload_widget = widgets.FileUpload(
    accept=".txt,.pdf",
    multiple=False,
    description="Choose file",
    layout=widgets.Layout(width="180px")
)
# Clicking this button triggers the actual reading and processing of the file
load_file_btn = widgets.Button(
    description="⬆ Load File",
    button_style="success",
    layout=widgets.Layout(width="120px")
)
upload_status = widgets.HTML("")   # Displays a success or error message after loading
pdf_preview   = widgets.HTML("")   # Displays the inline PDF preview iframe after loading

upload_panel = widgets.VBox(
    [widgets.HBox([upload_widget, load_file_btn]), upload_status, pdf_preview],
    layout=widgets.Layout(
        display="none",   # Hidden by default until user clicks "Upload File"
        padding="10px 14px",
        border="1px dashed #bbb",
        margin="4px 0"
    )
)

# ── TOGGLE LOGIC ──────────────────────────────────────────────────────────────
# Controls which panel is visible (paste or upload) and which button
# appears highlighted/active.

def set_active(active):
    if active == "paste":
        paste_panel.layout.display    = ""
        upload_panel.layout.display   = "none"
        btn_paste.style.button_color  = "#1a73e8"
        btn_upload.style.button_color = "#f0f0f0"
        btn_paste.style.text_color    = "white"
        btn_upload.style.text_color   = "#333"
    else:
        paste_panel.layout.display    = "none"
        upload_panel.layout.display   = ""
        btn_paste.style.button_color  = "#f0f0f0"
        btn_upload.style.button_color = "#1a73e8"
        btn_paste.style.text_color    = "#333"
        btn_upload.style.text_color   = "white"

# Default to paste mode when the tool first loads
set_active("paste")
btn_paste.on_click(lambda  _: set_active("paste"))
btn_upload.on_click(lambda _: set_active("upload"))

# ── FILE PROCESSING ───────────────────────────────────────────────────────────
# Runs when the user clicks "Load File".
# For PDFs: reads the file, extracts all text from every page,
#   shows a live inline PDF preview, and populates the transcript box.
# For TXT files: reads the raw text and populates the transcript box.
# After loading either type, automatically runs speaker detection.

def process_upload(b=None):
    if not upload_widget.value:
        upload_status.value = (
            "<span style='color:red;font-size:12px;'>⚠️ No file selected yet.</span>"
        )
        return

    # ipywidgets version 8.x stores uploaded files as a tuple of Bunch objects
    uploaded = upload_widget.value[0]
    filename = uploaded["name"]
    content  = bytes(uploaded["content"])  # Convert the raw file data to bytes

    if filename.endswith(".pdf"):
        try:
            import io, base64
            from pypdf import PdfReader
            # Encode the PDF as base64 text so it can be shown in an iframe preview
            b64    = base64.b64encode(content).decode("utf-8")
            reader = PdfReader(io.BytesIO(content))
            # Extract text from all pages and combine with line breaks
            text   = "\n".join(page.extract_text() or "" for page in reader.pages)
            transcript_box.value = text

            # Show a success message with a link to toggle the PDF preview
            upload_status.value = (
                f'<span style="color:green;font-size:12px;">✅ PDF loaded: <strong>{filename}</strong> '
                f'({len(reader.pages)} page{"s" if len(reader.pages)!=1 else ""}) — '
                f'{len(text.split()):,} words extracted — '
                f'<a href="#" onclick="'
                f'var box=document.getElementById(\'pdf-preview-box\');'
                f'box.style.display=box.style.display===\'none\'?\'block\':\'none\';'
                f'return false;" style="color:#1a73e8;">toggle preview</a></span>'
            )
            # Embed the actual PDF as a scrollable inline viewer
            pdf_preview.value = f"""
            <div id="pdf-preview-box"
                 style="margin-top:10px;border:1px solid #ddd;border-radius:6px;
                        overflow:hidden;display:block;">
              <div style="background:#f5f5f5;padding:6px 12px;font-size:12px;color:#555;
                          border-bottom:1px solid #ddd;display:flex;
                          justify-content:space-between;align-items:center;">
                <span>📄 {filename}</span>
                <span style="color:#888;">
                  {len(reader.pages)} page{"s" if len(reader.pages)!=1 else ""} •
                  {len(text.split()):,} words extracted
                </span>
              </div>
              <iframe src="data:application/pdf;base64,{b64}"
                      width="100%" height="400px"
                      style="display:block;border:none;"></iframe>
              <div style="padding:8px 12px;background:#e8f5e9;border-top:1px solid #c8e6c9;
                          font-size:12px;color:#2e7d32;">
                ✅ Text extracted and ready. Switch to
                <strong>Paste Text</strong> view to review before searching.
              </div>
            </div>"""
            # Automatically scan the extracted text for speaker labels
            refresh_speakers()

        except Exception as e:
            upload_status.value = (
                f'<span style="color:red;font-size:12px;">❌ Could not read PDF: {e}</span>'
            )
            pdf_preview.value = ""

    elif filename.endswith(".txt"):
        try:
            text = content.decode("utf-8")
            transcript_box.value = text
            upload_status.value  = (
                f'<span style="color:green;font-size:12px;">✅ Loaded: <strong>{filename}</strong> '
                f'— {len(text.split()):,} words</span>'
            )
            pdf_preview.value = ""
            set_active("paste")   # Switch to paste view so the user can see the loaded text
            refresh_speakers()
        except Exception as e:
            upload_status.value = (
                f'<span style="color:red;font-size:12px;">❌ Could not read file: {e}</span>'
            )
    else:
        upload_status.value = (
            "<span style='color:red;font-size:12px;'>⚠️ Please select a .txt or .pdf file.</span>"
        )

load_file_btn.on_click(process_upload)

# ── SPEAKER FILTER ────────────────────────────────────────────────────────────
# The section that lets researchers restrict the search to specific speakers.
# After clicking "Detect Speakers", a checkbox appears for each speaker found.
# Checking one or more means only those speakers' lines are searched.
# Leaving all unchecked searches the entire transcript.

speaker_label_widget = widgets.HTML("""
<div style="font-family:'Segoe UI',sans-serif;margin-top:4px;">
  <b style="font-size:14px;">🎙 Speaker Filter</b>
  <span style="font-size:12px;color:#888;margin-left:8px;">
    Click detect after pasting your transcript
  </span>
</div>
""")
speaker_box    = widgets.HBox([], layout=widgets.Layout(flex_wrap="wrap", gap="8px", margin="6px 0"))
speaker_status = widgets.HTML(
    "<span style='font-size:12px;color:#aaa;font-style:italic;'>"
    "Paste or upload a transcript, then click Detect Speakers.</span>"
)
detect_btn = widgets.Button(
    description="🔍 Detect Speakers",
    button_style="info",
    layout=widgets.Layout(width="170px", height="30px", margin="4px 0")
)
speaker_section = widgets.VBox([speaker_label_widget, detect_btn, speaker_box, speaker_status])

# Dictionary that maps each speaker's name to their checkbox widget
_speaker_checkboxes = {}

# ── SPEAKER DETECTION ─────────────────────────────────────────────────────────
# Scans each line of the transcript looking for Otter.ai-style speaker labels:
#   "Speaker 1  0:00" — a name followed by spaces and a timestamp
# Returns a sorted, deduplicated list of speaker names found.

def detect_speakers_from_transcript(transcript: str) -> list[str]:
    speakers = set()
    # Words that look like speaker labels but shouldn't be treated as one
    false_positives = {
        'http', 'https', 'note', 'edit', 'update', 'timestamp',
        'date', 'time', 'location', 'subject', 'from', 'to', 'cc',
        'transcribed by'
    }
    # Normalize different line ending styles (Windows vs Mac vs Unix)
    normalized = transcript.replace('\r\n', '\n').replace('\r', '\n')
    for line in normalized.split('\n'):
        line = line.strip()
        if not line:
            continue
        # Look for: word(s) starting with a letter → whitespace → timestamp like 0:00 or 11:34
        match = re.match(
            r'^([A-Za-z][A-Za-z0-9]*(?:[ ][A-Za-z0-9]+)*)\s+(\d{1,2}:\d{2})',
            line
        )
        if not match:
            continue
        label = match.group(1).strip()
        if not label or label.lower() in false_positives:
            continue
        if len(label) > 40:
            continue
        # Must have at least one word that's purely letters (not just numbers)
        if not any(w.isalpha() for w in label.split()):
            continue
        speakers.add(label)
    return sorted(speakers)

# ── SPEAKER CHECKBOX BUILDER ──────────────────────────────────────────────────
# Runs the detection above and builds a checkbox for each speaker found.
# Called automatically when a file is uploaded, or manually via the Detect button.

def refresh_speakers(b=None):
    transcript = transcript_box.value.strip()
    _speaker_checkboxes.clear()
    if not transcript:
        speaker_box.children = []
        speaker_status.value = (
            "<span style='font-size:12px;color:#aaa;font-style:italic;'>"
            "Paste or upload a transcript, then click Detect Speakers.</span>"
        )
        return
    speakers = detect_speakers_from_transcript(transcript)
    if not speakers:
        speaker_box.children = []
        speaker_status.value = (
            "<span style='font-size:12px;color:#e65100;font-style:italic;'>"
            "No speaker labels detected — searching full transcript.</span>"
        )
        return
    checkboxes = []
    for name in speakers:
        cb = widgets.Checkbox(
            value=False, description=name, indent=False,
            layout=widgets.Layout(width="auto")
        )
        _speaker_checkboxes[name] = cb
        checkboxes.append(cb)
    speaker_box.children = checkboxes
    speaker_status.value = (
        f"<span style='font-size:12px;color:#2e7d32;'>"
        f"✅ {len(speakers)} speaker{'s' if len(speakers)!=1 else ''} detected. "
        f"Leave all unchecked to search everyone.</span>"
    )

detect_btn.on_click(refresh_speakers)

# ── SEARCH DESCRIPTION BOX ────────────────────────────────────────────────────
# Where the researcher types what they're looking for — this can be vague
# ("participant seemed annoyed") or specific ("couldn't find the search bar").
# The AI interprets the meaning behind the description.

desc_label = widgets.HTML(
    "<b style='font-family:Segoe UI,sans-serif;font-size:14px;'>🔎 Describe what you're looking for</b>"
)
desc_box = widgets.Textarea(
    placeholder='e.g. "Participant seemed frustrated they couldn\'t find the search bar"',
    layout=widgets.Layout(width="100%", height="70px")
)

# ── MINIMUM MATCH SCORE SLIDER ────────────────────────────────────────────────
# Lets researchers filter out low-confidence results.
# Drag right to only see strong matches; leave at 0 to see everything.
# The hint text below the slider updates in real time as you drag.

score_hint = widgets.HTML("")

def update_score_hint(change):
    # Updates the descriptive text beneath the slider as the value changes
    v = min_score_slider.value
    if v == 0:
        color, text = "#888", "Showing all results regardless of score"
    elif v < 50:
        color, text = "#c62828", f"Showing weak matches and above (≥ {v})"
    elif v < 80:
        color, text = "#e65100", f"Showing moderate matches and above (≥ {v})"
    else:
        color, text = "#2e7d32", f"Showing strong matches only (≥ {v})"
    score_hint.value = f"<span style='font-size:12px;color:{color};'>{text}</span>"

min_score_slider = widgets.IntSlider(
    value=0, min=0, max=100, step=5,
    continuous_update=True,
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width="360px")
)
min_score_slider.observe(update_score_hint, names="value")
update_score_hint(None)  # Set the initial hint text when the tool loads

# The main action button — clicking this triggers the AI search
run_button = widgets.Button(
    description="Find Quotes", button_style="primary",
    icon="search", layout=widgets.Layout(width="160px", height="36px")
)

# ── TWO OUTPUT AREAS ──────────────────────────────────────────────────────────
# The tool uses two separate display areas so that running a new search
# doesn't wipe out the quote library.
# - results_output: cleared and redrawn every time Find Quotes is clicked
# - library_output: drawn once when first used, never touched again

results_output = widgets.Output()
library_output = widgets.Output()
_state         = {"transcript": "", "library_initialized": False}

# ── QUOTE LIBRARY ─────────────────────────────────────────────────────────────
# Builds the quote library panel the first time Find Quotes is clicked.
# After that it is never re-rendered, so folders and saved quotes survive
# across multiple searches. Only resets if the kernel is restarted.
# The library lets researchers:
#   - Create named folders (e.g. "Navigation Issues", "Positive Reactions")
#   - Drag result cards into folders to save quotes by theme
#   - Click "Jump to transcript" on any saved quote to locate it
#   - Export a folder's contents as a .txt file

def init_library():
    with library_output:
        display(HTML("""
<div id="quote-library-root"
     style="font-family:'Segoe UI',sans-serif;width:100%;
            background:#fff;border:1px solid #e0e0e0;border-radius:8px;
            overflow:hidden;display:flex;flex-direction:column;">
  <div style="background:#1a73e8;color:white;padding:10px 14px;
              display:flex;justify-content:space-between;align-items:center;">
    <span style="font-weight:700;font-size:14px;">📚 Quote Library</span>
    <button id="add-folder-btn"
            style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);
                   color:white;border-radius:4px;padding:3px 10px;
                   font-size:12px;cursor:pointer;">+ New Folder</button>
  </div>
  <div id="new-folder-row"
       style="display:none;padding:8px 10px;border-bottom:1px solid #eee;
              background:#f9f9f9;gap:6px;align-items:center;">
    <input id="folder-name-input" type="text" placeholder="Folder name…"
           style="flex:1;padding:5px 8px;border:1px solid #ccc;border-radius:4px;
                  font-size:13px;width:160px;"/>
    <button id="confirm-folder-btn"
            style="background:#1a73e8;color:white;border:none;border-radius:4px;
                   padding:5px 10px;font-size:12px;cursor:pointer;">Create</button>
    <button id="cancel-folder-btn"
            style="background:none;border:1px solid #ccc;border-radius:4px;
                   padding:5px 8px;font-size:12px;cursor:pointer;color:#666;">✕</button>
  </div>
  <div id="folder-list" style="overflow-y:auto;padding:8px;">
    <div id="library-placeholder"
         style="color:#aaa;font-size:12px;text-align:center;margin-top:20px;">
      Create a folder, then drag quotes into it.
    </div>
  </div>
</div>

<style>
  .folder-drop-zone {
    border:2px dashed #ccc;border-radius:6px;padding:8px;margin-bottom:10px;
    transition:border-color .15s,background .15s;
  }
  .folder-drop-zone.drag-over { border-color:#1a73e8;background:#e8f0fe; }
  .saved-quote-item {
    background:#f9f9f9;border:1px solid #eee;border-radius:6px;
    padding:8px 10px;margin-top:6px;font-size:12px;color:#333;position:relative;
  }
  .saved-quote-item .delete-quote {
    position:absolute;top:6px;right:8px;
    background:none;border:none;cursor:pointer;color:#aaa;font-size:13px;
  }
  .saved-quote-item .delete-quote:hover { color:#c62828; }
  .folder-header {
    display:flex;justify-content:space-between;align-items:center;
    font-weight:700;font-size:13px;color:#333;padding:4px 2px;
    cursor:pointer;user-select:none;
  }
  .folder-header .delete-folder {
    background:none;border:none;cursor:pointer;color:#aaa;font-size:12px;padding:0 4px;
  }
  .folder-header .delete-folder:hover { color:#c62828; }
  .export-btn {
    margin-top:6px;width:100%;background:none;border:1px solid #1a73e8;
    color:#1a73e8;border-radius:4px;padding:4px 0;font-size:11px;cursor:pointer;
  }
  .export-btn:hover { background:#e8f0fe; }
  .jump-from-library {
    font-size:10px;color:#1a73e8;cursor:pointer;text-decoration:underline;
  }
  .jump-from-library:hover { color:#0d47a1; }
</style>

<script>
(function() {

  // Scrolls the transcript pane to a specific line and highlights it in purple
  function jumpToLine(lineId) {
    document.querySelectorAll('.tline-highlight')
            .forEach(el => el.classList.remove('tline-highlight'));
    const target = document.getElementById(lineId);
    const pane   = document.getElementById('transcript-pane');
    if (target && pane) {
      target.classList.add('tline-highlight');
      pane.scrollTop = target.offsetTop - pane.offsetTop - 40;
    }
  }

  const addFolderBtn    = document.getElementById('add-folder-btn');
  const newFolderRow    = document.getElementById('new-folder-row');
  const folderNameInput = document.getElementById('folder-name-input');
  const confirmBtn      = document.getElementById('confirm-folder-btn');
  const cancelBtn       = document.getElementById('cancel-folder-btn');
  const folderList      = document.getElementById('folder-list');

  // Show or hide the folder name input row
  addFolderBtn.addEventListener('click', () => {
    newFolderRow.style.display = 'flex';
    folderNameInput.focus();
  });
  cancelBtn.addEventListener('click', () => {
    newFolderRow.style.display = 'none';
    folderNameInput.value = '';
  });

  // Fallback: if a card is dropped onto the library area but misses a folder,
  // add it to the first folder (or auto-create "My Quotes" if no folders exist)
  document.getElementById('quote-library-root').addEventListener('dragover', e => {
    e.preventDefault();
  });
  document.getElementById('quote-library-root').addEventListener('drop', e => {
    e.preventDefault();
    const raw = e.dataTransfer.getData('application/json');
    if (!raw) return;
    try {
      const data = JSON.parse(raw);
      if (!folderList.querySelector('.folder-drop-zone')) {
        _createFolder('My Quotes');
      }
      const first = folderList.querySelector('.folder-drop-zone');
      if (first) _addQuoteToFolder(first, data);
    } catch(err) {}
  });

  // Creates a new named folder in the library with drag-drop, export, and delete
  function _createFolder(name) {
    if (!name.trim()) return;
    const placeholder = document.getElementById('library-placeholder');
    if (placeholder) placeholder.remove();
    const folderId = 'folder-' + Date.now();
    const wrapper  = document.createElement('div');
    wrapper.className = 'folder-drop-zone';
    wrapper.id = folderId;
    wrapper.innerHTML = `
      <div class="folder-header">
        <span>📁 ${name}</span>
        <span style="display:flex;gap:4px;align-items:center;">
          <span class="folder-quote-count"
                style="font-size:11px;color:#888;font-weight:400;">0 quotes</span>
          <button class="delete-folder" title="Delete folder">✕</button>
        </span>
      </div>
      <div class="folder-quotes"></div>
      <button class="export-btn">⬇ Export as .txt</button>
    `;

    // Clicking the ✕ button deletes the folder and all quotes inside it
    wrapper.querySelector('.delete-folder').addEventListener('click', () => {
      if (confirm('Delete folder "' + name + '" and all its quotes?')) {
        wrapper.remove();
        if (!folderList.querySelector('.folder-drop-zone')) {
          folderList.innerHTML =
            '<div id="library-placeholder" style="color:#aaa;font-size:12px;' +
            'text-align:center;margin-top:20px;">Create a folder, then drag quotes into it.</div>';
        }
      }
    });

    // Clicking Export saves all quotes in the folder to a .txt file on the researcher's computer
    wrapper.querySelector('.export-btn').addEventListener('click', () => {
      const items = wrapper.querySelectorAll('.saved-quote-item');
      if (!items.length) { alert('No quotes in this folder yet.'); return; }
      let txt = 'Quote Library — ' + name + '\\n' + '='.repeat(40) + '\\n\\n';
      items.forEach((item, idx) => {
        const q  = item.getAttribute('data-quote');
        const ts = item.getAttribute('data-timestamp');
        const r  = item.getAttribute('data-reasoning');
        const s  = item.getAttribute('data-score');
        txt += (idx+1) + '. ';
        if (ts) txt += '[' + ts + '] ';
        txt += '"' + q + '"\\n';
        txt += '   Match score: ' + s + '/100\\n';
        txt += '   Why it matches: ' + r + '\\n\\n';
      });
      const blob = new Blob([txt], {type:'text/plain'});
      const a    = document.createElement('a');
      a.href     = URL.createObjectURL(blob);
      a.download = name.replace(/\\s+/g,'-') + '-quotes.txt';
      a.click();
    });

    // Visual feedback when a card is being dragged over this folder
    wrapper.addEventListener('dragover', e => {
      e.preventDefault();
      wrapper.classList.add('drag-over');
    });
    wrapper.addEventListener('dragleave', () => wrapper.classList.remove('drag-over'));

    // Accept the dropped card and add it to this folder
    // stopPropagation prevents the drop from also firing on the library root,
    // which would have caused the quote to be added twice
    wrapper.addEventListener('drop', e => {
      e.preventDefault();
      e.stopPropagation();
      wrapper.classList.remove('drag-over');
      const raw = e.dataTransfer.getData('application/json');
      if (!raw) return;
      try { _addQuoteToFolder(wrapper, JSON.parse(raw)); } catch(err) {}
    });

    folderList.appendChild(wrapper);
  }

  // Adds a quote card into a folder, with a jump-to-transcript link and a delete button
  function _addQuoteToFolder(folderEl, data) {
    const quotesContainer = folderEl.querySelector('.folder-quotes');
    const countEl         = folderEl.querySelector('.folder-quote-count');
    const item            = document.createElement('div');
    item.className = 'saved-quote-item';
    item.setAttribute('data-quote',     data.quote     || '');
    item.setAttribute('data-timestamp', data.timestamp || '');
    item.setAttribute('data-reasoning', data.reasoning || '');
    item.setAttribute('data-score',     data.score     || '');
    item.setAttribute('data-lineid',    data.lineid    || '');

    const tsText  = data.timestamp
      ? '<span style="color:#1565c0;font-size:10px;margin-right:4px;">⏱ ' + data.timestamp + '</span>'
      : '';
    // Show a preview of the quote (first 80 characters) to keep cards compact
    const preview = (data.quote||'').length > 80
      ? data.quote.substring(0,80) + '…' : data.quote;

    item.innerHTML = `
      ${tsText}
      <em style="line-height:1.5;display:block;margin-bottom:4px;">"${preview}"</em>
      <div style="display:flex;justify-content:space-between;align-items:center;margin-top:4px;">
        <span style="color:#888;font-size:11px;">Score: ${data.score}/100</span>
        ${data.lineid
          ? '<span class="jump-from-library">🔗 Jump to transcript</span>'
          : ''}
      </div>
      <button class="delete-quote" title="Remove">✕</button>
    `;

    // Clicking "Jump to transcript" scrolls the transcript pane to this quote's location
    if (data.lineid) {
      item.querySelector('.jump-from-library').addEventListener('click', e => {
        e.stopPropagation();
        jumpToLine(data.lineid);
      });
    }

    // Clicking ✕ removes just this one quote from the folder
    item.querySelector('.delete-quote').addEventListener('click', () => {
      item.remove();
      const rem = quotesContainer.querySelectorAll('.saved-quote-item').length;
      countEl.textContent = rem + ' quote' + (rem !== 1 ? 's' : '');
    });

    quotesContainer.appendChild(item);
    const total = quotesContainer.querySelectorAll('.saved-quote-item').length;
    countEl.textContent = total + ' quote' + (total !== 1 ? 's' : '');
  }

  // Create the folder when the user presses Create or hits Enter
  confirmBtn.addEventListener('click', () => {
    _createFolder(folderNameInput.value);
    folderNameInput.value      = '';
    newFolderRow.style.display = 'none';
  });
  folderNameInput.addEventListener('keydown', e => {
    if (e.key === 'Enter') confirmBtn.click();
  });

})();
</script>
"""))
    _state["library_initialized"] = True

# ── SCORE COLOR AND LABEL HELPERS ─────────────────────────────────────────────
# Small helper functions that return the right color and label text
# based on a numeric match score. Used to style the score bars on result cards.

def score_color(score):
    if score >= 80: return "#2e7d32"   # Green — strong match
    if score >= 55: return "#e65100"   # Amber — moderate match
    return "#c62828"                   # Red — weak match

def score_label_str(score):
    if score >= 80: return "Strong match"
    if score >= 55: return "Moderate match"
    return "Weak match"

# ── RESULTS RENDERER ──────────────────────────────────────────────────────────
# Builds the complete HTML for the two-column results view:
#   Left column: the full transcript in a scrollable dark-mode viewer
#   Right column: ranked result cards with score bars, key term highlights,
#                 expand/collapse, and drag handles for the quote library
# Also renders the blue informational banner about how scores work.

def render_results(results, description, transcript, selected_speakers):
    # Convert the transcript text into HTML where each line has its own ID
    # so the page can scroll to and highlight specific lines
    transcript_html = build_transcript_html(transcript)

    # Show which speakers were filtered to, or "all speakers" if none selected
    if selected_speakers:
        spk_text  = ", ".join(selected_speakers)
        spk_badge = (
            f'<div style="margin-bottom:8px;font-size:12px;color:#555;">'
            f'🎙 Filtered to: <strong>{spk_text}</strong></div>'
        )
    else:
        spk_badge = (
            '<div style="margin-bottom:8px;font-size:12px;color:#888;">'
            '🎙 All speakers searched</div>'
        )

    # Blue banner reminding researchers that scores are AI estimates, not exact measurements
    score_banner = """
    <div style="display:flex;align-items:flex-start;gap:10px;
                padding:10px 14px;margin-bottom:14px;
                background:#eef4fb;border:1px solid #c9dff0;
                border-left:3px solid #7aabce;
                border-radius:8px;font-size:11.5px;line-height:1.6;color:#4a6070;">
      <span style="font-size:16px;margin-top:1px;">ℹ️</span>
      <span style="font-family:'Jost',sans-serif;">
        <strong style="color:#2c3e50;font-weight:500;">About match scores</strong>
        &nbsp;—&nbsp;
        Scores are relative to each other within this search and reflect semantic
        relevance estimates generated by AI. They are not precise measurements —
        a 90 is a stronger match than a 60, but the same quote may score
        differently across separate searches. Use scores as a ranking guide,
        not an absolute value.
      </span>
    </div>
    """

    # If no quotes passed the minimum score threshold, show a helpful message
    if not results:
        cards_html = (
            '<div style="color:#888;font-size:13px;padding:20px;text-align:center;">'
            'No quotes met your minimum match score. Try lowering the threshold.</div>'
        )
    else:
        cards_html = ""
        for i, r in enumerate(results):
            quote     = r.get("quote", "")
            timestamp = r.get("timestamp")
            reasoning = r.get("reasoning", "")
            score     = int(r.get("match_score", 0))
            key_terms = r.get("key_terms", [])
            speaker   = r.get("speaker")

            # Find where this quote lives in the transcript for the jump-to feature
            offset  = find_quote_char_offset(transcript, quote)
            line_id = char_offset_to_line_id(transcript, offset)

            # Apply yellow highlights to key terms within the quote text
            highlighted  = highlight_terms(
                quote.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"),
                key_terms
            )
            # Default view shows a 120-character preview; full quote shown on expand
            preview_plain = quote[:120] + ("…" if len(quote) > 120 else "")
            preview_hl    = highlight_terms(
                preview_plain.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"),
                key_terms
            )

            # Blue timestamp badge in the card header (if transcript has timestamps)
            ts_badge = (
                f'<span style="background:#e3f2fd;color:#1565c0;font-size:11px;'
                f'padding:2px 8px;border-radius:10px;margin-left:6px;font-weight:600;">'
                f'⏱ {timestamp}</span>'
            ) if timestamp else ""

            # Green speaker badge in the card header
            spk_tag = (
                f'<span style="background:#e8f5e9;color:#2e7d32;font-size:11px;'
                f'padding:2px 8px;border-radius:10px;margin-left:6px;font-weight:600;">'
                f'🎙 {speaker}</span>'
            ) if speaker else ""

            sc_color = score_color(score)
            sc_lbl   = score_label_str(score)

            # Purple pill tags showing which specific words drove the match
            pills = "".join(
                f'<span style="background:#f3e5f5;color:#6a1b9a;font-size:11px;'
                f'padding:2px 7px;border-radius:10px;margin-right:4px;">{t}</span>'
                for t in key_terms
            )

            # Safely encode quote data into HTML attributes (avoids breaking the HTML)
            quote_attr = quote.replace('"','&quot;').replace('\n',' ')
            ts_attr    = (timestamp or "").replace('"','&quot;')
            r_attr     = reasoning.replace('"','&quot;')
            li_attr    = line_id or ""

            # Package the card's data as JSON so it can be passed to the library when dragged
            drag_json  = json.dumps({
                "quote":     quote_attr,
                "timestamp": timestamp or "",
                "reasoning": reasoning,
                "score":     str(score),
                "lineid":    li_attr
            }).replace('"', '&quot;')

            cards_html += f"""
            <div class="qcard" id="qcard-{i}"
                 draggable="true"
                 data-quote="{quote_attr}"
                 data-timestamp="{ts_attr}"
                 data-reasoning="{r_attr}"
                 data-score="{score}"
                 data-lineid="{li_attr}"
                 data-drag="{drag_json}"
                 style="background:#fff;border:1px solid #e0e0e0;border-radius:8px;
                        margin-bottom:14px;overflow:hidden;
                        cursor:{'pointer' if line_id else 'grab'};
                        transition:box-shadow .15s;">
              <div style="display:flex;align-items:center;justify-content:space-between;
                          padding:10px 14px;background:#fafafa;border-bottom:1px solid #eee;">
                <span style="font-weight:700;color:#333;font-size:13px;">
                  <span style="color:#aaa;margin-right:6px;cursor:grab;"
                        title="Drag to library">⠿</span>
                  Match #{i+1}{ts_badge}{spk_tag}
                </span>
                <span style="display:flex;align-items:center;gap:6px;">
                  <span style="font-size:12px;color:{sc_color};font-weight:600;">{sc_lbl}</span>
                  <span style="display:inline-block;width:80px;height:8px;background:#eee;
                               border-radius:4px;overflow:hidden;">
                    <span style="display:block;width:{score}%;height:100%;
                                 background:{sc_color};border-radius:4px;"></span>
                  </span>
                  <span style="font-size:12px;font-weight:700;color:{sc_color};">{score}/100</span>
                </span>
              </div>
              <div class="preview-text"
                   style="padding:12px 14px 4px;font-style:italic;color:#222;
                          font-size:14px;line-height:1.7;">"{preview_hl}"</div>
              <div class="full-text"
                   style="display:none;padding:0 14px 4px;font-style:italic;color:#222;
                          font-size:14px;line-height:1.7;">"{highlighted}"</div>
              <div style="padding:6px 14px 12px;display:flex;flex-direction:column;gap:6px;">
                {'<button class="toggle-btn" style="align-self:flex-start;background:none;border:1px solid #bbb;border-radius:4px;padding:3px 10px;font-size:12px;cursor:pointer;color:#555;">Show full quote ▾</button>' if len(quote) > 120 else ''}
                <div style="font-size:12px;color:#666;">
                  <strong>Why this matches:</strong> {reasoning}
                </div>
                {'<div style="margin-top:4px;"><strong style="font-size:11px;color:#888;">Key terms: </strong>' + pills + '</div>' if pills else ''}
                {'<div style="font-size:11px;color:#1a73e8;margin-top:4px;">🔗 Click card to locate in transcript →</div>' if line_id else ''}
              </div>
            </div>"""

    return f"""
<div style="font-family:'Segoe UI',sans-serif;display:flex;gap:16px;align-items:flex-start;">

  <!-- Left column: the transcript text with each line tagged for jump-to scrolling -->
  <div style="flex:1;min-width:0;">
    <div style="font-weight:700;font-size:13px;color:#555;margin-bottom:6px;
                letter-spacing:.04em;">TRANSCRIPT</div>
    <div id="transcript-pane"
         style="height:540px;overflow-y:auto;background:#1e1e2e;color:#cdd6f4;
                font-family:'Courier New',monospace;font-size:12.5px;line-height:1.8;
                padding:14px;border-radius:8px;border:1px solid #313244;
                white-space:pre-wrap;">
      {transcript_html}
    </div>
  </div>

  <!-- Right column: the ranked result cards -->
  <div style="flex:1;min-width:0;">
    <div style="font-weight:700;font-size:13px;color:#555;margin-bottom:4px;
                letter-spacing:.04em;">
      RESULTS FOR: <em style="font-weight:400;">"{description}"</em>
    </div>
    {spk_badge}
    <div id="results-pane" style="max-height:540px;overflow-y:auto;">
      {score_banner}
      {cards_html}
    </div>
  </div>
</div>

<style>
  .qcard:hover {{ box-shadow:0 2px 12px rgba(0,0,0,.12); }}
  .qcard[draggable="true"]:active {{ opacity:0.6; }}
  #transcript-pane span.tline-highlight {{
    background:#45475a;outline:2px solid #cba6f7;border-radius:2px;
  }}
</style>

<script>
(function() {{

  // Removes the old highlight and scrolls to the new target line in the transcript
  function jumpToLine(lineId) {{
    document.querySelectorAll('.tline-highlight')
            .forEach(el => el.classList.remove('tline-highlight'));
    const target = document.getElementById(lineId);
    const pane   = document.getElementById('transcript-pane');
    if (target && pane) {{
      target.classList.add('tline-highlight');
      pane.scrollTop = target.offsetTop - pane.offsetTop - 40;
    }}
  }}

  // Toggle button inside a card: shows/hides the full quote text
  document.querySelectorAll('.toggle-btn').forEach(btn => {{
    btn.addEventListener('click', function(e) {{
      e.stopPropagation();
      const card    = btn.closest('.qcard');
      const preview = card.querySelector('.preview-text');
      const full    = card.querySelector('.full-text');
      const expanded = full.style.display !== 'none';
      preview.style.display = expanded ? '' : 'none';
      full.style.display    = expanded ? 'none' : '';
      btn.textContent       = expanded ? 'Show full quote ▾' : 'Collapse ▴';
    }});
  }});

  // Clicking a result card jumps to that quote's location in the transcript
  document.querySelectorAll('.qcard').forEach(card => {{
    card.addEventListener('click', function(e) {{
      if (e.target.classList.contains('toggle-btn')) return;
      const lineId = card.getAttribute('data-lineid');
      if (lineId) jumpToLine(lineId);
    }});
  }});

  // When a card is dragged, attach its data as JSON so the library can receive it
  document.querySelectorAll('.qcard').forEach(card => {{
    card.addEventListener('dragstart', function(e) {{
      const raw = card.getAttribute('data-drag');
      if (raw) {{
        const decoded = raw.replace(/&quot;/g, '"');
        e.dataTransfer.setData('application/json', decoded);
        e.dataTransfer.effectAllowed = 'copy';
      }}
    }});
  }});

}})();
</script>
"""

# ── FIND QUOTES BUTTON HANDLER ────────────────────────────────────────────────
# Everything that happens when the researcher clicks "Find Quotes":
#   1. Saves the current transcript to memory
#   2. Initializes the quote library (first time only)
#   3. Validates that a transcript and description are present
#   4. Reads which speakers are checked and what score threshold is set
#   5. Calls the AI search function and displays the results
# If anything goes wrong, shows a clear error message.

def on_run_clicked(b):
    _state["transcript"] = transcript_box.value.strip()

    # Build the quote library panel the very first time Find Quotes is clicked
    if not _state["library_initialized"]:
        init_library()

    with results_output:
        clear_output(wait=True)
        transcript  = _state["transcript"]
        description = desc_box.value.strip()

        if not transcript:
            display(HTML(
                "<p style='color:red;font-family:Segoe UI,sans-serif;'>"
                "⚠️ Please paste or upload a transcript first.</p>"
            ))
            return
        if not description:
            display(HTML(
                "<p style='color:red;font-family:Segoe UI,sans-serif;'>"
                "⚠️ Please describe what you're looking for.</p>"
            ))
            return

        # Gather which speaker checkboxes the researcher has ticked
        selected_speakers = [
            name for name, cb in _speaker_checkboxes.items() if cb.value
        ]
        min_score = min_score_slider.value
        spk_msg   = (
            f"speakers: {', '.join(selected_speakers)}"
            if selected_speakers else "all speakers"
        )

        # Show a "searching..." message while the API call is in progress
        display(HTML(
            f"<p style='color:#555;font-family:Segoe UI,sans-serif;'>"
            f"⏳ Analyzing transcript ({spk_msg}, min score: {min_score})…</p>"
        ))

        try:
            results = extract_quotes_from_transcript(
                transcript, description,
                min_score=min_score,
                selected_speakers=selected_speakers
            )
            clear_output(wait=True)
            display(HTML(render_results(results, description, transcript, selected_speakers)))
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"<p style='color:red;'>❌ Error: {e}</p>"))

run_button.on_click(on_run_clicked)

# ── FINAL PAGE LAYOUT ─────────────────────────────────────────────────────────
# Stacks all the UI sections into a single vertical column and displays them.
# The results and quote library sit together at the bottom in their own area
# so the library can persist across searches without being cleared.

results_and_library = widgets.VBox(
    [results_output, library_output],
    layout=widgets.Layout(gap="16px")
)

display(widgets.VBox([
    transcript_label,       # Privacy warning + "Interview Transcript" label
    toggle_row,             # Paste Text / Upload File toggle buttons
    paste_panel,            # The transcript text area (paste mode)
    upload_panel,           # The file picker + load button (upload mode)
    widgets.HTML("<br>"),
    speaker_section,        # Speaker filter checkboxes
    widgets.HTML("<br>"),
    desc_label, desc_box,   # "Describe what you're looking for" input
    widgets.HTML("<br>"),
    widgets.HTML("<b style='font-family:Segoe UI,sans-serif;'>Minimum match score to show:</b>"),
    min_score_slider,       # Score threshold slider (0–100)
    score_hint,             # Dynamic hint text beneath the slider
    widgets.HTML("<br>"),
    run_button,             # "Find Quotes" button
    widgets.HTML("<br>"),
    results_and_library     # Search results + persistent quote library
], layout=widgets.Layout(padding="10px", max_width="1400px")))
